In [20]:
pip install optimum-intel

   ---------------------------------------- 0.0/721.1 kB ? eta -:--:--
   -------------- ------------------------- 262.1/721.1 kB ? eta -:--:--
   ---------------------------------------- 721.1/721.1 kB 2.3 MB/s  0:00:00
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.2 MB 4.6 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/10.2 MB 7.7 MB/s eta 0:00:01
   ---------------------- ----------------- 5.8/10.2 MB 9.3 MB/s eta 0:00:01
   ----------------------------------- ---- 9.2/10.2 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 10.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 17.6 MB/s  0:00:00
   ---------------------------------------- 0.0/795.0 kB ? eta -:--:--
   ---------------------------------------- 795.0/795.0 kB 17.2 MB/s  0:00:00
   -----------------------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [21]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "llama3.2"

DB_NAME = "preprocessed_db"
collection_name = "docs"
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(type(embedding_model))
KNOWLEDGE_BASE_PATH = Path("telecom_demo_kb_large/documents")
AVERAGE_CHUNK_SIZE = 500



d:\moved\projects\RAG_implementation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3813.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 580 documents


In [6]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: alarm_guides
The document has been retrieved from: telecom_demo_kb_large/documents/alarm_guides/alarm_guide_001.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 3 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Alarm Guide: 

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: alarm_guides\nThe document has been retrieved from: telecom_demo_kb_large/documents/alarm_guides/alarm_guide_001.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 3 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with ove

In [10]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=f"ollama/{MODEL}",messages=messages,response_format=Chunks,api_base="http://localhost:11434")
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [11]:
process_document(documents[0])

[Result(page_content='Alarm meaning\n\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer.\n\n# Alarm Guide: BGP-NEIGHBOR-FLAP\n\n## Alarm meaning\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.', metadata={'source': 'telecom_demo_kb_large/documents/alarm_guides/alarm_guide_001.md', 'type': 'alarm_guides'}),
 Result(page_content='What commonly correlates\n\nRelated alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## What commonly correlates\nRelated alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n\n', metadata={'source': 'telecom_demo_kb_large/documents/alarm_guides/alarm_guide_001.md', 'type': 'alarm_guides'}),
 Result(page_content='False positives

In [12]:
def create_chunks(documents):
    chunks = []

    for doc in tqdm(documents[:30]):
        chunks.extend(process_document(doc))

    return chunks

In [13]:
chunks = create_chunks(documents)

100%|██████████| 30/30 [35:45<00:00, 71.50s/it]


In [17]:
print(len(chunks))

121


In [22]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = embedding_model.encode(texts)
    vectors = emb.tolist()

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [23]:
create_embeddings(chunks)

Vectorstore created with 121 documents


In [26]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
color_map = {
    'historical_incidents': 'blue',
    'postmortems': 'green',
    'runbooks': 'red',
    'known_errors': 'orange',
    'vendor_troubleshooting': 'purple',
    'alarm_guides': 'brown',
    'change_context': 'pink',
    'performance_baselines': 'gray',
    'site_operations': 'cyan',
    'topology_context': 'magenta'
}

colors = [color_map[t] for t in doc_types]

In [27]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [28]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [36]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(
        model=f"ollama/{MODEL}",
        messages=messages,
        response_format=RankOrder,
        api_base="http://localhost:11434"
    )
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [32]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = embedding_model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [33]:
question = "What alarm is associated with BGP peer?"
chunks = fetch_context_unranked(question)

In [34]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Alarm meaning a...
Alarm meaning

...
Alarm Meaning

...
Alarm meaning

...
Alarm Meaning

...
Alarm meaning

...
What commonly c...
What commonly c...
What commonly c...
What commonly c...


In [37]:
reranked = rerank(question, chunks)

[1, 3, 7, 9, 10, 2, 4, 6, 5, 8]


In [38]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Alarm meaning a...
Alarm Meaning

...
What commonly c...
What commonly c...
What commonly c...
Alarm meaning

...
Alarm meaning

...
Alarm meaning

...
Alarm Meaning

...
What commonly c...


In [39]:
question = "What is HIGH-CPU alarm"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [42]:
reranked = rerank(question, chunks[:10])

[5, 1, 2, 4, 8, 10, 9, 6, 7]


In [43]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)


In [44]:
reranked[0].page_content

'False Positives and Secondary Symptoms\n\nUnderstanding when the HIGH-CPU alarm may appear as a false positive or secondary symptom.\n\n## False positives and secondary symptoms\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n\n## Recommended context\nAn RCA agent should retrieve the affected device, interface or peer, site, service dependencies, current telemetry, recent changes, and historical incidents matching the alarm and platform.'

In [45]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [46]:
SYSTEM_PROMPT_TEMPLATE = """You are an AI-assisted Root Cause Analysis (RCA) Agent for a telecom Network Operations Center (NOC).

Your primary responsibility is to analyze network and IT incidents using the provided knowledge base, incident information, 
alarms, telemetry, logs, topology context, historical incidents, postmortems, runbooks, known errors, vendor troubleshooting guides,
 change records, and performance baselines. Politely decline irrelevant question and do not answer it. IF there is not relevant context in the knowladge base. say so instead of making assumptions
 Relevant context:
 {context}
 """

In [48]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [55]:

def rewrite_query(question, history=[]):

    """Rewrite the user's question into a concise, specific Knowledge Base search query for telecom IT/network RCA."""

    message = f"""
You are an AI assistant supporting a telecom Network Operations Center (NOC) and IT operations team.

Your job is to rewrite the user's question into a short, precise search query that will be used to retrieve relevant information from a Knowledge Base.

The Knowledge Base contains information such as:
- Historical network and IT incidents
- Root cause analyses and postmortems
- Network alarms and alarm meanings
- Known errors and failure patterns
- Troubleshooting and operational runbooks
- Router and switch incidents
- Server and infrastructure incidents
- Network topology and device dependencies
- Site and POP information
- Configuration and change history
- Performance baselines and anomalies
- Vendor troubleshooting information
- Previous incident resolutions

Conversation history:
{history}

Current user question:
{question}

Rewrite the question into ONE very short and specific Knowledge Base search query.

Focus on the important technical details, such as:
- Device or network element
- Site or POP
- Alarm or error
- Service or interface
- Failure symptom
- Time or incident context
- Dependency or affected component
- Possible root-cause-related terms

Preserve important technical terminology, device names, alarm names, error codes, site names, and identifiers from the user's question.

If the user's question refers to something mentioned in the conversation history, use the history to make the rewritten query self-contained.

Do not invent information that is not present in the question or conversation history.

Remove unnecessary conversational wording.

Do not answer the question.
Do not explain your reasoning.
Do not mention the Knowledge Base.
Do not mention that you rewrote the question.

IMPORTANT:
Respond ONLY with the final search query.
The output must be a single short query, not a paragraph.
"""

    response = completion(
        model=f"ollama/{MODEL}",
        messages=[
            {"role": "user", "content": message}
        ],
        api_base="http://localhost:11434"
    )

    return response.choices[0].message.content.strip()


In [58]:
rewrite_query("What alarm is associated with BGP peer?elaborate what kind", [])

'"BGP peer - BGP neighbor down"'

In [61]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(
    model=f"ollama/{MODEL}",
    messages=messages,
    api_base="http://localhost:11434"
    )   
    return response.choices[0].message.content, chunks

In [62]:
answer_question("What alarm is associated with BGP peer?elaborate what kind of alarm", [])

`BGP peer down - ELS-00001 - NNC-POP-XYZ`
[14, 11, 16, 18, 20, 13, 19, 17, 15, 10, 12, 6, 5, 3, 2, 1]


("The alarm associated with BGP peer is BGP-NEIGHBOR-DOWN. This is a synthetic major alarm, which means it's an observation rather than an automatic root-cause classification. It indicates that there's an issue with the BGP neighbor, but it doesn't necessarily mean there's a specific root cause for the issue.",
 [Result(page_content='What commonly correlates\n\nRelated alarms may include ISIS-ADJ-DOWN, BGP-NEIGHBOR-DOWN, BGP-NEIGHBOR-FLAP. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## What commonly correlates\nRelated alarms may include ISIS-ADJ-DOWN, BGP-NEIGHBOR-DOWN, BGP-NEIGHBOR-FLAP. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## False positives and secondary symptoms\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n', metad

In [63]:
answer_question("What is HIGH-CPU alarm", [])

HIGH-CPU alarm (Router: Core Routers, Site: NYS-01)
[-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


('The HIGH-CPU alarm is a synthetic major alarm associated with a device. It should be interpreted as an observation, not an automatic root-cause classification. This alarm commonly correlates with other alarms such as LINK-DOWN, CRC-ERRORS, and BGP-ROUTE-THRESHOLD. Correlation across time and topology can reveal whether the alarm is primary or downstream.',
 [Result(page_content='False positives and secondary symptoms\n\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault.\n\n## False positives and secondary symptoms\n\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n', metadata={'type': 'alarm_guides', 'source': 'telecom_demo_kb_large/documents/alarm_guides/alarm_guide_027.md'}),
  Result(page_content='False Positives and Secondary Symptoms\n\nUnderstanding when the HIG